In [ ]:


# CONFIGURAÇÕES INICIAIS DAS ANÁLISES (PRESENTES EM TODOS OS SCRIPTS)
# IMPORTAR BIBLIOTECAS ---
import sys
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# CAMINHOS ---
SCRIPT_DIR = Path(__file__).resolve().parent        # caminho desse script
ANALYTICS_DIR = SCRIPT_DIR.parent                   # pasta desse script

# PARA IMPORTAR FUNÇÕES DE EXTRAIR CSV ---
if str(ANALYTICS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYTICS_DIR))
from utils.export_utils import exportar_csv


# ROOT DO PROJETO ---
PROJECT_ROOT = Path(__file__).resolve().parents[3]

# ROOT DOS OUTPUTS ---
OUTPUT_DIR = (PROJECT_ROOT/ "scripts"/ "Analytics"/ "outputs"/ "gold_01")
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# SPARK ---
spark = (
    SparkSession.builder
    .appName("TechChallenge_Analytics")
    .master("local[*]")
    .getOrCreate()
)
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# PREPARAÇÃO PARA ANALISAR GOLD 01 - Como está estruturado o mercado brasileiro de Dados? ---
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# CAMINHO DO ARQUIVO ---
caminho_gold_01 = (PROJECT_ROOT/ "Gold"/ "perguntas_negocio"/ "gold_01_estrutura_mercado")

# BUSCA CSVs GERADOS PELO SPARK NA CRIAÇÃO DA GOLD ---
arquivos_gold_01 = [
    str(arquivo)
    for arquivo in caminho_gold_01.glob("part-*.csv")
]
print("Arquivos encontrados:")
print(arquivos_gold_01)

# CARREGAR GOLD 01 ---
df_estrutura = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(arquivos_gold_01)
)
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# INÍCIO DAS ANÁLISES ---
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# SETOR ---
# ANÁLISE INICIAL DA DIMENSÃO ---
df_setor = (df_estrutura
    .filter(F.col("variavel") == "setor")
    .orderBy("edicao", F.desc("pct_na_dimensao"))
)
df_setor.show(100, truncate=False)
print('Qtd linhas', df_setor.count())

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
A análise inicial da dimensão de setor identificou 63 registros,
correspondentes às categorias de setor distribuídas entre as três
edições da pesquisa.

O número utilizado como base para cálculo dos percentuais varia
entre as edições:

- 2023-2024: 4.753 respondentes
- 2024-2025: 4.863 respondentes
- 2025-2026: 3.228 respondentes

Por esse motivo, as comparações entre os períodos serão realizadas
principalmente por participação percentual, e não por contagem
absoluta.

A inspeção inicial também permite verificar quais setores possuem
maior representatividade em cada edição e se existem mudanças na
taxonomia que possam afetar a comparação histórica.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# LISTA DE SETORES EXISTENTES POR EDIÇÃO ---
(df_setor
    .select("edicao","valor")
    .distinct()
    .orderBy("edicao","valor")
    .show(200, truncate=False)
)

# QUANTIDADE DE SETORES POR EDIÇÃO ---
(df_setor
    .groupBy("edicao")
    .agg(F.countDistinct("valor").alias("qtd_setores"))
    .orderBy("edicao")
    .show(truncate=False)
)

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
A validação identificou 21 categorias de setor em cada uma das
três edições analisadas.

Além da mesma quantidade de categorias, a inspeção dos valores
mostra que os setores utilizados permanecem consistentes entre
os períodos.

Dessa forma, diferentemente de outras dimensões que apresentaram
mudanças de taxonomia, a dimensão de setor permite comparação
histórica direta entre 2023-2024, 2024-2025 e 2025-2026.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# COMPARATIVO HISTÓRICO DOS SETORES ---
comparativo_setor = (df_setor
    .groupBy("valor")
    .pivot("edicao",["2023-2024", "2024-2025", "2025-2026"])
    .agg(F.first("pct_na_dimensao"))
)
comparativo_setor.show(100, truncate=False)

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
O comparativo histórico mostra que Finanças ou Bancos e
Tecnologia/Fábrica de Software permanecem como os dois setores
com maior participação nas três edições.

Finanças ou Bancos apresenta:

- 2023-2024: 19,5%
- 2024-2025: 21,3%
- 2025-2026: 18,5%

Tecnologia/Fábrica de Software apresenta:

- 2023-2024: 18,0%
- 2024-2025: 19,4%
- 2025-2026: 17,4%

Embora ambos apresentem aumento intermediário em 2024-2025,
a participação retorna a níveis menores na edição mais recente.

Outros setores apresentam participações individualmente menores,
resultando em uma distribuição relativamente pulverizada após
os dois principais grupos.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# VARIAÇÃO ENTRE A PRIMEIRA E A ÚLTIMA EDIÇÃO ---
comparativo_setor = (comparativo_setor
    .withColumn("variacao_pp",F.round(F.col("2025-2026") - F.col("2023-2024"),2))
    .orderBy(F.desc("variacao_pp"))
)
comparativo_setor.show(100, truncate=False)

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
A comparação entre a primeira e a última edição mostra que as
mudanças de participação são relativamente moderadas na maioria
dos setores.

O maior aumento de participação ocorre em Indústria, que passa
de 6,6% em 2023-2024 para 7,4% em 2025-2026, uma variação de
+0,8 ponto percentual.

Internet/Ecommerce, Área de Consultoria e Educação apresentam
aumento de +0,6 p.p. cada.

A maior redução ocorre em Varejo, cuja participação passa de
8,2% para 5,6%, uma diferença de -2,6 pontos percentuais.

Finanças ou Bancos apresenta redução de -1,0 p.p., enquanto
Tecnologia/Fábrica de Software registra -0,6 p.p.

PONTO DE ATENÇÃO:
As variações representam mudanças na composição dos respondentes
das pesquisas e não permitem concluir, isoladamente, que determinado
setor cresceu ou diminuiu no mercado brasileiro.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# CONCENTRAÇÃO DOS DOIS PRINCIPAIS SETORES - 2025-2026 ---
top2_setores_atual = (df_setor
    .filter(F.col("edicao") == "2025-2026")
    .orderBy(F.desc("pct_na_dimensao"))
    .limit(2)
    .agg(F.round(F.sum("pct_na_dimensao"),2).alias("participacao_top2"))
)
top2_setores_atual.show(truncate=False)

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
Na edição 2025-2026, os dois setores com maior participação são
Finanças ou Bancos e Tecnologia/Fábrica de Software.

Juntos, esses dois setores concentram 35,9% dos respondentes da
dimensão de setor.

Isso significa que pouco mais de um terço da amostra atual está
concentrado nesses dois segmentos, enquanto os demais respondentes
se distribuem entre outras 19 categorias de setor.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# ESTRUTURA ATUAL DOS SETORES - 2025-2026 ---
setor_atual = (df_setor
    .filter(F.col("edicao") == "2025-2026")
    .select("valor","contagem","pct_na_dimensao")
    .orderBy(F.desc("pct_na_dimensao"))
)
setor_atual.show(100, truncate=False)

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
Na edição mais recente, Finanças ou Bancos permanece como o setor
com maior participação na amostra, representando 18,5% dos
respondentes.

Na sequência aparecem:

- Tecnologia/Fábrica de Software: 17,4%
- Área de Consultoria: 8,8%
- Outra Opção: 7,5%
- Indústria: 7,4%
- Varejo: 5,6%
- Educação: 5,0%

A partir do terceiro colocado, nenhum setor individual ultrapassa
10% de participação.

Esse resultado mostra que, embora Finanças e Tecnologia concentrem
uma parcela relevante da amostra, existe presença de profissionais
de Dados distribuída entre diversos setores econômicos.

Os percentuais representam a composição dos respondentes da pesquisa
e não devem ser interpretados isoladamente como a distribuição de
profissionais de Dados em todo o mercado brasileiro.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# EXPORTAÇÃO DOS RESULTADOS PARA VISUALIZAÇÃO ---
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# SETOR ATUAL.CSV
exportar_csv(setor_atual,OUTPUT_DIR,"setor_atual.csv")

# COMPARAÇÃO HISTÓRICA.CSV
exportar_csv(comparativo_setor,OUTPUT_DIR,"setor_historico.csv")